# Kaggle Run: Lightweight V1 Confidence-Gated RAG

This notebook unpacks the attached project dataset into `/kaggle/working`, then runs the smoke test and inspects outputs.


In [ ]:
# Notebook parameters — edit these with your Kaggle slugs or absolute input paths
# If you supply absolute /kaggle/input paths, set the slug vars to None so the notebook
# will use the attached files instead of calling the Kaggle CLI.
github_repo_url = ''  # e.g. 'https://github.com/your-user/your-repo.git'
github_repo_branch = 'main'
dataset_slug = None
model_dataset_slug = None
model_path_within_dataset = ''
auto_run_experiment = False

# Absolute paths (set one or both as needed):
input_root_override = '/kaggle/input/datasets/buildformacarov/squad-20'  # <- your dataset path
model_absolute_path = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1'  # <- your model file or folder


In [ ]:
import os, tarfile, shutil, subprocess
from pathlib import Path

input_root_override = globals().get('input_root_override', None)
model_absolute_path = globals().get('model_absolute_path', None)
github_repo_url = globals().get('github_repo_url', None)
github_repo_branch = globals().get('github_repo_branch', 'main')
dataset_slug = globals().get('dataset_slug', None)
model_dataset_slug = globals().get('model_dataset_slug', None)
repo_root = None
working_root = Path('/kaggle/working')

# If a GitHub repo is provided, clone it into /kaggle/working/repo and run from there.
if github_repo_url:
    repo_root = Path('/kaggle/working/repo')
    if repo_root.exists():
        shutil.rmtree(repo_root)
    print('Cloning GitHub repo:', github_repo_url)
    cmd = ['git', 'clone', '--branch', github_repo_branch, '--depth', '1', github_repo_url, str(repo_root)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout or res.stderr)
    if res.returncode != 0:
        raise RuntimeError('git clone failed')
    os.chdir(repo_root)
    working_root = repo_root

# Prefer explicit /kaggle/input override when provided.
if input_root_override:
    input_root = Path(input_root_override)
elif dataset_slug:
    input_root = Path('/kaggle/input') / dataset_slug
else:
    input_root = Path('/kaggle/input') / 'kausikvaibhavpatra/kaggle-rag-v1-code'

if not input_root.exists():
    # Fallback: find the first attached dataset that contains our files
    candidates = [p for p in Path('/kaggle/input').iterdir() if p.is_dir()]
    for cand in candidates:
        if (cand / 'run_experiment.py').exists() or (cand / 'src.tar').exists():
            input_root = cand
            break

print('Using dataset root:', input_root)
os.chdir(working_root)

# Copy root-level files
for name in ['run_experiment.py', 'config.yaml', 'requirements.txt', 'requirements-py311.txt', 'README.md']:
    src = input_root / name
    if src.exists():
        shutil.copy2(src, working_root / name)

# Extract any tar archives that hold folders
for tar_name in ['src.tar', 'scripts.tar']:
    tar_path = input_root / tar_name
    if tar_path.exists():
        with tarfile.open(tar_path) as tf:
            tf.extractall(working_root)

# If an absolute model path is provided, surface it in the working dir when possible
if model_absolute_path:
    model_path = Path(model_absolute_path)
    print('Using model path:', model_path)
    if model_path.exists() and model_path.is_file():
        shutil.copy2(model_path, working_root / model_path.name)
    elif model_path.exists() and model_path.is_dir():
        for p in model_path.iterdir():
            if p.is_file():
                shutil.copy2(p, working_root / p.name)

print('Working files:', sorted([p.name for p in working_root.iterdir() if p.is_file()])[:20])
print('src exists:', (working_root / 'src').exists())
print('scripts exists:', (working_root / 'scripts').exists())


In [ ]:
import sys, platform, os
print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('CWD:', os.getcwd())
try:
    import torch
    print('Torch available, CUDA:', torch.cuda.is_available())
except Exception as e:
    print('Torch not available:', e)


In [ ]:
import subprocess, sys
cmd = [sys.executable, 'run_experiment.py', '--mode', 'smoke']
print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', proc.returncode)
print('STDOUT:\n', proc.stdout[:2000])
print('STDERR:\n', proc.stderr[:2000])


In [ ]:
import json
from pathlib import Path
import pandas as pd
pred_path = Path('outputs/predictions/predictions.csv')
metrics_path = Path('outputs/metrics/metrics.json')
if pred_path.exists():
    display(pd.read_csv(pred_path).head())
else:
    print('Predictions file not found:', pred_path)
if metrics_path.exists():
    print(json.load(open(metrics_path)))
else:
    print('Metrics file not found:', metrics_path)
